<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/retrievers/memory_aware_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Memory-Aware Retriever (Mini Example)

This minimal example shows how to wrap an existing retriever so each query is enriched with recalled agent memory.

Use this when you want retrieval to reflect session context (for example, a remembered product tier or region) without storing that context inside the document index.

In [ ]:
%pip install llama-index-core

In [ ]:
from typing import List, Optional

from llama_index.core import Document, VectorStoreIndex
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.llms import ChatMessage
from llama_index.core.memory import Memory, StaticMemoryBlock
from llama_index.core.schema import NodeWithScore, QueryBundle


class MemoryAwareRetriever(BaseRetriever):
    """Prepends recalled memory to the query before delegating retrieval."""

    def __init__(self, base_retriever: BaseRetriever, memory: Memory) -> None:
        self._base_retriever = base_retriever
        self._memory = memory
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        memory_messages = self._memory.get(
            messages=[ChatMessage(role="user", content=query_bundle.query_str)]
        )
        memory_text = " ".join(m.content for m in memory_messages if m.content)
        enriched_query = (
            f"{memory_text} {query_bundle.query_str}".strip()
            if memory_text
            else query_bundle.query_str
        )
        return self._base_retriever.retrieve(enriched_query)

    async def _aretrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        memory_messages = await self._memory.aget(
            messages=[ChatMessage(role="user", content=query_bundle.query_str)]
        )
        memory_text = " ".join(m.content for m in memory_messages if m.content)
        enriched_query = (
            f"{memory_text} {query_bundle.query_str}".strip()
            if memory_text
            else query_bundle.query_str
        )
        return await self._base_retriever.aretrieve(enriched_query)

## Try it on a tiny index

In [ ]:
docs = [
    Document(text="Enterprise SLA: 99.95% uptime with 1-hour response time."),
    Document(text="Starter SLA: best-effort support with 2-business-day response time."),
]
index = VectorStoreIndex.from_documents(docs)
base_retriever = index.as_retriever(similarity_top_k=1)

memory = Memory.from_defaults(
    session_id="retriever-demo",
    memory_blocks=[
        StaticMemoryBlock(
            name="plan",
            static_content="The user is on the Enterprise plan.",
            priority=0,
        )
    ],
)

retriever = MemoryAwareRetriever(base_retriever=base_retriever, memory=memory)
nodes = retriever.retrieve("What SLA response time applies to me?")
print(nodes[0].node.get_content())

Without memory, the retriever might return the Starter SLA. After memory enrichment, the query is biased toward Enterprise context. For production systems, prefer explicit metadata filters when possible; use this wrapper for lightweight session-aware retrieval.